# Stage 3B — DM-Count clean-protocol smoke

This notebook freezes a deterministic 240/60 split inside ShanghaiTech Part A `train_data`, runs one clean smoke epoch, validates only on the 60-image training-derived partition, and writes a 100-point review bundle to Google Drive. It does not evaluate `test_data`.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/DroneAI')
PART_A = DRIVE_ROOT / 'datasets' / 'shanghaitech' / 'extracted' / 'ShanghaiTech' / 'part_A'
RUN_DIR = DRIVE_ROOT / 'runs' / 'stage-3' / 'clean-smoke' / 'seed-2026'
assert PART_A.is_dir(), PART_A
print({'data': str(PART_A), 'run_dir': str(RUN_DIR)})


In [ ]:
import os
import stat
import subprocess
from google.colab import userdata

REPO_URL = 'https://github.com/LuciTa81/DroneAI.git'
REPO_DIR = Path('/content/DroneAI')
BRANCH = 'agent/stage3-dm-count-reproduction'
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Add the GITHUB_TOKEN secret and grant notebook access.')

askpass = Path('/tmp/droneai_git_askpass.py')
askpass.write_text(
    "#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'password' in prompt else 'x-access-token')\n",
    encoding='utf-8',
)
askpass.chmod(askpass.stat().st_mode | stat.S_IEXEC)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': token})
try:
    if (REPO_DIR / '.git').is_dir():
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], env=git_env, check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], env=git_env, check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], env=git_env, check=True)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], env=git_env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    del token
    git_env.pop('GITHUB_TOKEN', None)

subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--short', '--branch'], check=True)


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)


In [ ]:
WORK_DIR = Path('/content/droneai-stage3b/seed-2026')
cmd = [
    sys.executable, '-u', str(REPO_DIR / 'scripts' / 'run_stage3b_clean_smoke.py'),
    '--data-dir', str(PART_A),
    '--run-dir', str(RUN_DIR),
    '--work-dir', str(WORK_DIR),
    '--upstream-dir', '/content/DM-Count',
    '--seed', '2026',
    '--smoke-epochs', '1',
    '--num-workers', '0',
]
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

score = json.loads((RUN_DIR / 'score.json').read_text(encoding='utf-8'))
display(Markdown((RUN_DIR / 'score.md').read_text(encoding='utf-8')))
display(pd.DataFrame(score['checks'])[['category', 'description', 'weight', 'earned', 'blocker', 'observed']])
print('Decision:', score['status'], score['score'], '/ 100')
print('Artifacts:', RUN_DIR)
